# Generating an alias dictionary for SchemaMapper

The **alias dictionary** is a many-to-one mapping from real-world column
names to standard field names (e.g. `age_at_diagnosis` &larr; `AGE_D`,
`AGE_AT_DX`, `INVASIVE_CARCINOMA_DX_AGE`). Stage&nbsp;1 of
`SchemaMapEngine` matches incoming columns against it **before** any
embedding work, so it is the single biggest lever on SchemaMapper
accuracy. When you map to your own schema you will usually want to supply
a matching alias dictionary too.

This notebook walks the full lifecycle:

1. **define** the target (standardized) field names
2. **generate** aliases for them with an LLM &rarr; a DataFrame
3. **persist** it as `field_name,source,is_numeric_field` CSV
4. **consume** it via `SchemaMapEngine(alias_dict_path=...)`

> ⚠️ **This notebook spends LLM tokens.** Every run of the *generate*
> cell issues real API calls. The demo below uses a deliberately small
> 5-field target set so a run costs pennies — don't point it at a large
> schema without expecting the bill. Re-run the generate cell only when
> you actually want to regenerate.

> **Format reference:** [`../docs/input_formats.md`](../docs/input_formats.md) §2.5.
> A non-interactive CLI equivalent of this notebook lives alongside it as
> [`generate_alias_dict.py`](generate_alias_dict.py).

## 0. Setup

Pick a model; the provider is auto-detected from the prefix
(`claude-*` &rarr; Anthropic, `gemini-*` &rarr; Gemini) and the matching
key is read from the environment (`ANTHROPIC_API_KEY` /
`GEMINI_API_KEY`). Set it before launching Jupyter, or uncomment the line
below.

In [ ]:
import os
import pandas as pd

from metaharmonizer.models.schema_mapper.generate_alias_dict import generate_alias_dict

# Provider is auto-detected from the model prefix.
MODEL = "gemini-2.5-pro"   # or e.g. "claude-sonnet-4-6"

# os.environ["GEMINI_API_KEY"] = "..."          # or set before launching Jupyter
# os.environ["ANTHROPIC_API_KEY"] = "sk-..."

OUT_PATH = "data/outputs/alias_dict.csv"

## 1. Define the target attributes

These are the **standardized field names** you want aliases for. The
`is_numeric_field` flag is forwarded into the output and lets numeric
fields (ages, doses) be handled distinctly.

We use a small, **deliberately diverse** 5-field set so the output spans
every prompt pass — a numeric field, value-encoded categoricals, an
anatomical field, and a disease field — rather than five look-alike age
columns:

| field | exercises |
|---|---|
| `age_at_diagnosis` | abbreviation / composite (`AGE_D`, `AGE_AT_DX`) |
| `sex` | value-encoded (`GENDER`, `M/F`, `1/2`) |
| `vital_status` | status vocab (`alive/dead`, `0/1`) |
| `body_site` | anatomical synonyms (`SITE`, `ANATOMIC_SITE`) |
| `disease` | institutional / composite (`DIAGNOSIS`, `DX`) |

To generate for the **full bundled schema** instead, swap the cell for:
```python
from metaharmonizer.models.schema_mapper.config import CURATED_DICT_PATH
target_fields = pd.read_csv(CURATED_DICT_PATH)   # all 33 curated fields
```
or point at your own CSV with a `field_name` column.

In [ ]:
target_fields = pd.DataFrame(
    {
        "field_name": [
            "age_at_diagnosis",
            "sex",
            "vital_status",
            "body_site",
            "disease",
        ],
        "is_numeric_field": ["yes", "no", "no", "no", "no"],
    }
)
target_fields

## 2. Generate aliases (LLM call)

`generate_alias_dict()` runs a multi-pass prompt pipeline (synonym,
abbreviation, value-encoded, composite, institutional) per
`schema_domain`. With only 5 fields they all fit in one batch, so this is
**one API call per pass**. Returns a DataFrame with exactly
`field_name,source,is_numeric_field`.

> This is the cell that spends tokens.

In [ ]:
alias_df = generate_alias_dict(
    target_fields=target_fields,
    model=MODEL,
    schema_domain="cancer_genomics",
    # api_key=...,  # optional; falls back to the provider env var
)

print(f"{len(alias_df)} alias rows, "
      f"{alias_df['field_name'].nunique()}/{len(target_fields)} fields covered")
alias_df.head(20)

In [ ]:
# How many aliases did each target field get?
alias_df.groupby("field_name")["source"].count().sort_values(ascending=False)

## 3. Persist the dictionary

Write it in the canonical 3-column schema — the only columns
`DictLoader.load_alias_dict` reads. This CSV is the artifact you reuse;
you don't regenerate per run.

In [ ]:
from pathlib import Path

Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
alias_df.to_csv(OUT_PATH, index=False)
print(f"wrote {len(alias_df)} rows -> {OUT_PATH}")

## 4. Consume it in the engine

Point `SchemaMapEngine` at the generated CSV via `alias_dict_path=` and
confirm an alias-y input column now maps through Stage&nbsp;1. We feed a
real alias drawn from what we just generated, so the demo reflects the
actual output.

> Needs the ML extras (`sentence-transformers`, etc.). `mode="manual"`
> runs Stages&nbsp;1–3 and needs no LLM key.

In [ ]:
from metaharmonizer.models.schema_mapper.engine import SchemaMapEngine

sample_alias = str(alias_df["source"].iloc[0])
expected_field = str(alias_df["field_name"].iloc[0])
print(f"feeding column {sample_alias!r} (should map to {expected_field!r})")

demo_input = "data/outputs/_alias_demo_input.csv"
pd.DataFrame({sample_alias: ["x"]}).to_csv(demo_input, index=False)

engine = SchemaMapEngine(
    clinical_data_path=demo_input,
    mode="manual",
    top_k=3,
    alias_dict_path=OUT_PATH,
)
result = engine.run_schema_mapping()
result[[c for c in ('query', 'stage', 'method', 'match1') if c in result.columns]]